In [ ]:
# Import dependencies and load environment variables

import kagglehub
import os
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from dotenv import load_dotenv
load_dotenv()


In [ ]:
# Download the latest dataset version
path = kagglehub.dataset_download("harshadapatil31/student-performance-and-study-habits-dataset")

print("Path to dataset files:", path)

files = os.listdir(path)

csv_files = []
for file in files:
    if file.lower().endswith('.csv'):
        csv_files.append(file)

if not csv_files:
    raise FileNotFoundError('No CSV file found in the downloaded dataset')

csv_path = os.path.join(path, csv_files[0])

df = pd.read_csv(csv_path)

In [ ]:
print(f'Head:\n{df.head()}\n')

print(f'Describe:\n{df.describe()}\n')

In [ ]:

print(f'dfisnull: {df.isnull().sum()}\n')

print(df.dtypes)

print(f'dfshape: {df.shape}\n')

print(f'dfcolumns: {df.columns}\n')

print(f'duplicated rows: {df.duplicated().sum()}\n')

The initial data-quality checks found missing values in `parental_education` and no duplicated rows.

## Handling missing values

There are three possible approaches to the missing values in `parental_education`:

1. Drop the entire column.
2. Replace missing values with the column mode.
3. Replace missing values with a new category.

There are 102 missing values, approximately 10% of the dataset. Because the missingness itself may carry information, the Silver layer preserves those records and uses `Unknown` as a separate category.

In [ ]:
df['parental_education'] = df['parental_education'].fillna('Unknown')

In [ ]:
categorical_columns = [
    'gender',
    'parental_education',
    'internet_access',
    'extracurricular_activities',
    'part_time_job',
    'final_grade'
]

for column in categorical_columns:
    df[column] = df[column].str.strip().str.title()
    print(column, df[column].unique(), '\n')

# final_grade is an ordinal categorical variable

The following assertions validate the cleaned Silver dataset against its expected schema and business rules.

In [ ]:
df = df.drop_duplicates()

assert df.duplicated().sum() == 0
assert df['student_id'].notna().all()
assert df['student_id'].is_unique
assert df['attendance_percent'].between(0, 100).all()
assert df['study_time_hours'].ge(0).all()
assert df['sleep_hours'].ge(0).all()
assert df['previous_grade'].between(0, 100).all()
assert df['final_exam_score'].between(0, 100).all()
assert df['parental_education'].notna().all()
assert df.isna().sum().sum() == 0

assert set(df['internet_access'].unique()) <= {'Yes', 'No'}
assert set(df['extracurricular_activities'].unique()) <= {'Yes', 'No'}
assert set(df['part_time_job'].unique()) <= {'Yes', 'No'}
assert set(df['final_grade'].unique()) <= {'A', 'B', 'C', 'D', 'F'}

In [ ]:
os.makedirs('./data/silver', exist_ok=True)

df.to_parquet('./data/silver/student_habits_silver.parquet', index=False)

print('Silver layer saved successfully')

# Start of the gold layer

Converting Yes/No to boolean

In [ ]:
gold_df = pd.read_parquet(
    "./data/silver/student_habits_silver.parquet"
).copy()

In [ ]:
def convert_to_boolean(df, column) -> pd.DataFrame:
    values = (
        df[column]
        .str.strip()
        .str.lower()
    )

    invalid = set(values.dropna()) - {"yes", "no"}

    if invalid:
        raise ValueError(
            f"Invalid values in {column}: {invalid}"
        )

    df[column] = values.map({
        "yes": True,
        "no": False
    })

    return df

In [ ]:
gold_df = convert_to_boolean(gold_df, "internet_access")
gold_df = convert_to_boolean(gold_df, "extracurricular_activities")
gold_df = convert_to_boolean(gold_df, "part_time_job")

In [ ]:
numeric_columns = [
    "student_id",
    "study_time_hours",
    "attendance_percent",
    "sleep_hours",
    "previous_grade",
    "final_exam_score",
]

gold_df[numeric_columns] = gold_df[numeric_columns].apply(
    pd.to_numeric
)

assert gold_df["student_id"].is_unique
assert gold_df["final_grade"].isin(["A", "B", "C", "D", "F"]).all()

## Creating the dims and the fact table

In [ ]:
dim_gender = gold_df[["gender"]].drop_duplicates()

dim_education = (
    gold_df[["parental_education"]]
    .drop_duplicates()
)

dim_student = gold_df[
    [
        "student_id",
        "gender",
        "parental_education",
        "internet_access",
        "extracurricular_activities",
        "part_time_job",
    ]
].drop_duplicates("student_id")

fact_student_performance = gold_df[
    [
        "student_id",
        "study_time_hours",
        "attendance_percent",
        "sleep_hours",
        "previous_grade",
        "final_exam_score",
        "final_grade",
    ]
]

In [ ]:
url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
    host=os.getenv("POSTGRES_HOST"),
    port=os.getenv("POSTGRES_PORT"),
    database=os.getenv("POSTGRES_DB")
)

engine = create_engine(url)

print(url)

In [ ]:
gold_df.to_sql(
    "stg_student_habits",
    engine,
    if_exists="replace",
    index=False
)

with engine.begin() as connection:
    connection.execute(text("""
        INSERT INTO dim_gender (gender)
        SELECT DISTINCT gender
        FROM stg_student_habits
        ON CONFLICT (gender) DO NOTHING;

        INSERT INTO dim_education (parental_education)
        SELECT DISTINCT parental_education
        FROM stg_student_habits
        ON CONFLICT (parental_education) DO NOTHING;

        INSERT INTO dim_student (
            student_id,
            gender_key,
            education_key,
            internet_access,
            extracurricular_activities,
            part_time_job
        )
        SELECT
            s.student_id,
            g.gender_key,
            e.education_key,
            s.internet_access,
            s.extracurricular_activities,
            s.part_time_job
        FROM stg_student_habits s
        JOIN dim_gender g
            ON g.gender = s.gender
        JOIN dim_education e
            ON e.parental_education = s.parental_education
        ON CONFLICT (student_id) DO NOTHING;

        INSERT INTO fact_student_performance (
            student_key,
            study_time_hours,
            attendance_percent,
            sleep_hours,
            previous_grade,
            final_exam_score
        )
        SELECT
            d.student_key,
            s.study_time_hours,
            s.attendance_percent,
            s.sleep_hours,
            s.previous_grade,
            s.final_exam_score
        FROM stg_student_habits s
        JOIN dim_student d
            ON d.student_id = s.student_id
        ON CONFLICT (student_key) DO NOTHING;
    """))